# ConsumerLens

In [1]:
import pandas as pd

In [2]:
# df = pd.read_csv("amazon_reviews.csv")

df = pd.read_csv(
    "../data/amazon_reviews.csv",
    engine="python",
    on_bad_lines="skip"
)


In [3]:
print(df.columns)
print(df.shape)
print(df.head())
print(df.isnull().sum())

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')
(21214, 9)
      Reviewer Name                     Profile Link Country Review Count  \
0        Eugene ath  /users/66e8185ff1598352d6b3701a      US     1 review   
1  Daniel ohalloran  /users/5d75e460200c1f6a6373648c      GB    9 reviews   
2          p fisher  /users/546cfcf1000064000197b88f      GB   90 reviews   
3         Greg Dunn  /users/62c35cdbacc0ea0012ccaffa      AU    5 reviews   
4     Sheila Hannah  /users/5ddbe429478d88251550610e      GB    8 reviews   

                Review Date                  Rating  \
0  2024-09-16T13:44:26.000Z  Rated 1 out of 5 stars   
1  2024-09-16T18:26:46.000Z  Rated 1 out of 5 stars   
2  2024-09-16T21:47:39.000Z  Rated 1 out of 5 stars   
3  2024-09-17T07:15:49.000Z  Rated 1 out of 5 stars   
4  2024-09-16T18:37:17.000Z  Rated 1 out of 5 stars   

             

In [4]:
print(df.shape)
print(df.columns)

(21214, 9)
Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')


In [5]:
df = df.rename(columns={
    "Country": "country",
    "Rating": "rating",
    "Review Title": "review_title",
    "Review Text": "review_text",
    "Date of Experience": "experience_date"
})

In [6]:
df["review_id"] = range(1, len(df) + 1)

In [7]:
df["experience_date"] = pd.to_datetime(
    df["experience_date"],
    errors="coerce"
)

df = df.dropna(subset=["experience_date"])

In [8]:
df["year_month"] = df["experience_date"].dt.to_period("M").astype(str)

In [9]:
print(df["rating"].unique())

['Rated 1 out of 5 stars' 'Rated 5 out of 5 stars'
 'Rated 2 out of 5 stars' 'Rated 4 out of 5 stars'
 'Rated 3 out of 5 stars']


In [10]:
df["rating"] = df["rating"].str.extract(r"(\d+)").astype(float)

In [11]:
df = df[[
    "review_id",
    "country",
    "rating",
    "review_title",
    "review_text",
    "experience_date",
    "year_month"
]]

In [12]:
print(df.shape)
print(df["rating"].describe())
print(df["country"].value_counts().head())
print(df["year_month"].value_counts().head())

(20947, 7)
count    20947.000000
mean         2.174440
std          1.671204
min          1.000000
25%          1.000000
50%          1.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64
country
US    9260
GB    7238
CA     706
IN     628
IE     241
Name: count, dtype: int64
year_month
2022-12    425
2019-12    390
2023-12    382
2024-07    378
2024-08    367
Name: count, dtype: int64


In [13]:
import sqlite3

conn = sqlite3.connect("consumerlens.db")

df.to_sql("reviews", conn, if_exists="replace", index=False)

conn.close()

In [14]:
conn = sqlite3.connect("consumerlens.db")
cursor = conn.cursor()

cursor.execute("""
SELECT country, AVG(rating)
FROM reviews
GROUP BY country
ORDER BY AVG(rating) ASC;
""")

print(cursor.fetchall())
conn.close()

[('AF', 1.0), ('AG', 1.0), ('BM', 1.0), ('BO', 1.0), ('BQ', 1.0), ('BW', 1.0), ('BZ', 1.0), ('CD', 1.0), ('CW', 1.0), ('GF', 1.0), ('GY', 1.0), ('IM', 1.0), ('IQ', 1.0), ('KG', 1.0), ('LA', 1.0), ('MN', 1.0), ('MQ', 1.0), ('MU', 1.0), ('PG', 1.0), ('TT', 1.0), ('UY', 1.0), ('VI', 1.0), ('LV', 1.2222222222222223), ('PA', 1.3333333333333333), ('PR', 1.3636363636363635), ('CY', 1.375), ('CR', 1.5), ('HU', 1.5454545454545454), ('ZA', 1.5769230769230769), ('TW', 1.7142857142857142), ('BG', 1.7272727272727273), ('CA', 1.8229461756373937), ('NL', 1.8726415094339623), ('IE', 1.8755186721991701), ('GE', 1.9090909090909092), ('AU', 1.934782608695652), ('TH', 1.9375), ('NZ', 1.9655172413793103), ('NO', 1.9767441860465116), ('AO', 2.0), ('BY', 2.0), ('FJ', 2.0), ('IS', 2.0), ('QA', 2.0), ('SR', 2.0), ('AE', 2.0224719101123596), ('CZ', 2.0434782608695654), ('SE', 2.0638297872340425), ('HR', 2.0714285714285716), ('US', 2.0989200863930884), ('LT', 2.1), ('DK', 2.1313559322033897), ('GB', 2.1798839458

### THE END